# ATMS 523
## Module 4 Lecture 2

### Using a Web API to obtain data

This example shows you how to use a web API to obtain data from NCEI.  It is the same dataset we obtained within Module 3, but here we are making small calls to get only the data we want.

In [6]:
#needed to make web requests
import requests

#store the data we get as a dataframe
import pandas as pd

#convert the response as a strcuctured json
import json

#mathematical operations on lists
import numpy as np

#parse the datetimes we get from NOAA
from datetime import datetime

#import time for sleeping
import time

#add the access token you got from NOAA # this is snesbitt's so don't mess with it!
# Token = 'qGhKPoTezWscLZFQAzmnmBzBkrsiwroe'
Token = 'uVuxRVewqamPjSlfmiBKiSjhBmGAQUsz'

#Enter data type and station ID from inventory - here is GHCND https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt
station_id = 'GHCND:USC00118740'

In [7]:
# #initialize lists to store data
# dates_mintemp = []
# dates_maxtemp = []
# dates_precip = []
# min_temps = []
# max_temps = []
# precip = []

# #for each year from 1905-2022 where we know we have data inventory ...
# for year in range(1905, 2023):
#     year = str(year)
#     print('working on year '+year)
    
#     #make the api call
#     r = requests.get('https://www.ncdc.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&datatypeid=TMIN&datatypeid=TMAX&datatypeid=PRCP&limit=1000&stationid='+station_id+'&startdate='+year+'-01-01&enddate='+year+'-12-31', headers={'token':Token})
#     #load the api response as a json
#     d = json.loads(r.text)
    

#     #get all items in the response which are max&min temperature readings
#     maxtemps = [item for item in d['results'] if item['datatype']=='TMAX']
#     mintemps = [item for item in d['results'] if item['datatype']=='TMIN']
#     precips = [item for item in d['results'] if item['datatype']=='PRCP']
#     #get the date field from all average temperature readings
#     dates_maxtemp += [item['date'] for item in maxtemps]
#     dates_mintemp += [item['date'] for item in mintemps]
#     dates_precip += [item['date'] for item in precips]
#     #get the actual temperature from the returned data
#     max_temps += [item['value'] for item in maxtemps]
#     min_temps += [item['value'] for item in mintemps]
#     precip += [item['value'] for item in precips]
#     time.sleep(0.2) # API max 5 requests per second


In [8]:
import requests, json, time

# Initialize lists
dates_mintemp, dates_maxtemp, dates_precip = [], [], []
min_temps, max_temps, precip = [], [], []

for year in range(1905, 2023):
    year = str(year)
    print(f"Working on year {year}...")

    base_url = (
        "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
        f"?datasetid=GHCND&datatypeid=TMIN&datatypeid=TMAX&datatypeid=PRCP"
        f"&limit=1000&stationid={station_id}&startdate={year}-01-01&enddate={year}-12-31"
    )

    offset = 1
    while True:
        url = base_url + f"&offset={offset}"
        r = requests.get(url, headers={"token": Token})
        
        # Handle HTTP errors
        if r.status_code != 200:
            print(f"  ⚠️ Skipping {year} (HTTP {r.status_code})")
            break

        try:
            d = r.json()
        except json.JSONDecodeError:
            print(f"  ⚠️ Invalid JSON for {year}, skipping.")
            break

        # No data
        if "results" not in d:
            print(f"  ⚠️ No data found for {year}")
            break

        # Extract data
        maxtemps = [item for item in d["results"] if item["datatype"] == "TMAX"]
        mintemps = [item for item in d["results"] if item["datatype"] == "TMIN"]
        precips  = [item for item in d["results"] if item["datatype"] == "PRCP"]

        dates_maxtemp += [item["date"] for item in maxtemps]
        dates_mintemp += [item["date"] for item in mintemps]
        dates_precip  += [item["date"] for item in precips]

        max_temps += [item["value"] for item in maxtemps]
        min_temps += [item["value"] for item in mintemps]
        precip    += [item["value"] for item in precips]

        # Check if there's more data
        if "metadata" in d and "resultset" in d["metadata"]:
            total = d["metadata"]["resultset"]["count"]
            if offset + 1000 > total:
                break
            offset += 1000
        else:
            break

        time.sleep(0.2)  # polite API rate limiting


Working on year 1905...
Working on year 1906...
Working on year 1907...
Working on year 1908...
Working on year 1909...
Working on year 1910...
Working on year 1911...
Working on year 1912...
Working on year 1913...
Working on year 1914...
Working on year 1915...
Working on year 1916...
Working on year 1917...
Working on year 1918...
Working on year 1919...
Working on year 1920...
Working on year 1921...
Working on year 1922...
Working on year 1923...
Working on year 1924...
Working on year 1925...
Working on year 1926...
Working on year 1927...
Working on year 1928...
Working on year 1929...
Working on year 1930...
Working on year 1931...
Working on year 1932...
Working on year 1933...
Working on year 1934...
Working on year 1935...
Working on year 1936...
Working on year 1937...
Working on year 1938...
Working on year 1939...
Working on year 1940...
Working on year 1941...
Working on year 1942...
Working on year 1943...
Working on year 1944...
Working on year 1945...
Working on year 

In [9]:
#initialize dataframe
df_temp_min = pd.DataFrame()
df_temp_max = pd.DataFrame()
df_precip = pd.DataFrame()

#populate date and min and max temperature & precip fields (convert string date to datetime)
df_temp_min['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_mintemp]
df_temp_min['minTemp'] = [float(v)/10.0 for v in min_temps]

df_temp_max['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_maxtemp]
df_temp_max['maxTemp'] = [float(v)/10.0 for v in max_temps]

df_precip['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_precip]
df_precip['precip'] = [float(v)/10.0 for v in precip]


In [10]:
#merge the dataframes
newdf_all = pd.merge(df_temp_max,df_temp_min, left_index=True, right_index=True)
newdf_all = pd.merge(newdf_all, df_precip, left_index=True, right_index=True)
newdf_all.drop(columns = ['date_x', 'date_y'], inplace=True)
newdf_all = newdf_all.set_index('date')

In [11]:
newdf_all

,maxTemp,minTemp,precip
date,,,
1905-01-01,14.4,1.7,0.0
1905-01-02,8.3,-5.0,4.8
1905-01-03,-3.9,-9.4,0.0
1905-01-04,-3.9,-14.4,0.0
1905-01-05,1.1,-14.4,0.0
...,...,...,...
2022-11-22,-7.2,-13.3,0.0
2022-11-23,-4.4,-12.8,0.0
2022-11-24,-2.2,-2.2,0.0
